# CatBoost GPU experiments — VS Code connected to Colab

This is a separate CatBoost workflow. The existing LightGBM notebooks are unchanged.
Connect this notebook to **Colab → New Colab Server → GPU**. Notebook cells run remotely;
the PowerShell terminal stays on your laptop. Upload `catboost_workflow/source.zip` with
VS Code Explorer's **Upload to Colab** command. You already have an uploaded source ZIP
for the older workflow; upload this separate ZIP too.

Run cells in order. The first full run is validation only. It cannot submit accidentally.
For a direct comparison, retain the July 2024 cutoff and no-promotion historical features.
Previous LightGBM validation AUC was 0.686407; reported Kaggle public score was 0.5656.
We cannot infer why those differ without auditing the exact submitted CSV.

In [3]:
import platform, subprocess

assert platform.system() == "Linux", "Select the remote Colab kernel, not local Windows Python"
subprocess.run(["nvidia-smi"], check=True)
print("Run cells in this remote notebook; local PowerShell does not use this GPU.")

Run cells in this remote notebook; local PowerShell does not use this GPU.


In [4]:
from pathlib import Path
import hashlib, os, subprocess, sys, zipfile

SOURCE_ZIP = Path("/content/source.zip")  # Change if Colab Contents shows another path.
if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"Upload catboost_workflow/source.zip to Colab, then set SOURCE_ZIP. Missing: {SOURCE_ZIP}")
digest = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
REPO = Path(f"/content/OptimlCatBoost_{digest[:10]}")
marker = REPO / ".source_zip.sha256"
if REPO.exists():
    if not marker.exists() or marker.read_text() != digest:
        raise RuntimeError("Source changed: choose a fresh REPO path or restart the Colab runtime before extracting.")
else:
    REPO.mkdir(parents=True)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for item in archive.infolist():
            if not (REPO / item.filename).resolve().is_relative_to(REPO.resolve()):
                raise ValueError("Unsafe ZIP path")
        archive.extractall(REPO)
    marker.write_text(digest)
assert (REPO / "optiml/gpu_check.py").is_file()
os.chdir(REPO)
print("CatBoost source ready:", REPO)

CatBoost source ready: /content/OptimlCatBoost_53e064c286


## Install and verify CatBoost GPU

This installs CatBoost and the data libraries. It does not install or compile LightGBM.
The preflight trains a tiny CatBoost model on the selected GPU and fails if the GPU is unavailable.

In [5]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "catboost_workflow/requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "optiml.gpu_check", "--model", "catboost", "--device", "GPU"], cwd=REPO, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'optiml.gpu_check', '--model', 'catboost', '--device', 'GPU'], returncode=0)

## Mount Drive and prepare the seven competition CSVs

Keep the original CSVs in `MyDrive/OptimlData`. Preparation runs once per Colab runtime.
CatBoost runs back up to `MyDrive/OptimlRuns/CatBoost/jay`, separate from other models.

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, shutil, json, uuid
from google.colab import drive

if "REPO" not in globals():
    raise RuntimeError("Run the source setup cell first")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
drive.mount("/content/drive")
DRIVE_DATA = Path("/content/drive/MyDrive/OptimlData")
DATA = Path("/content/optiml_data")
PREPARED = Path("/content/optiml_prepared/catboost-baseline-v1")
OWNER = "jay"
DRIVE_RUNS = Path("/content/drive/MyDrive/OptimlRuns/CatBoost") / OWNER
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
NAMES = ["train", "test", "customers", "products", "stores", "calendar", "sample_submission"]
for name in NAMES:
    source = DRIVE_DATA / f"{name}.csv"
    if not source.is_file():
        raise FileNotFoundError(f"Missing {source}; place all seven original CSVs in DRIVE_DATA")
    shutil.copy2(source, DATA / source.name)
print("Local CSVs ready:", DATA)

Mounted at /content/drive
Local CSVs ready: /content/optiml_data


In [7]:
from optiml.prepare import sha256

feature_source_hash = sha256(REPO / "optiml/features.py")
cache_manifest = PREPARED / "manifest.json"
cache_code = PREPARED / "feature_source.sha256"
if cache_manifest.exists():
    manifest = json.loads(cache_manifest.read_text())
    current = {name: sha256(DATA / f"{name}.csv") for name in NAMES}
    if manifest["source_sha256"] != current or not cache_code.exists() or cache_code.read_text() != feature_source_hash:
        raise ValueError("Data/features changed: choose a new PREPARED path and rerun preparation")
    print("Using matching completed cache:", PREPARED)
else:
    subprocess.run([sys.executable, "-m", "optiml.prepare", "--data-dir", str(DATA),
                    "--output-dir", str(PREPARED)], cwd=REPO, check=True)
    cache_code.write_text(feature_source_hash)

## Audit the earlier public-score submission when its exact CSV is available

This optional audit checks alignment against the prior run. A leaderboard gap has no known
cause until the submitted file and its generating run are compared.

In [9]:
PREVIOUS_RUN = None  # Example: Path("/content/drive/MyDrive/OptimlRuns/jay/<your_final_run>")
SUBMITTED_CSV = None # Path to the exact CSV used for the 0.5656 Kaggle submission.
if PREVIOUS_RUN is not None and SUBMITTED_CSV is not None:
    subprocess.run([sys.executable, "-m", "optiml.audit_submission", "--csv", str(SUBMITTED_CSV),
                    "--run-dir", str(PREVIOUS_RUN), "--sample", str(DATA / "sample_submission.csv")], cwd=REPO, check=True)
else:
    print("Audit pending: set PREVIOUS_RUN and SUBMITTED_CSV when available.")

Audit pending: set PREVIOUS_RUN and SUBMITTED_CSV when available.


## CatBoost experiment runner

Every fit explicitly requests GPU. Logs and results are backed up to Drive. Full validation
runs use the same held-out order lines, allowing an ID-hash check across configurations.

In [9]:
def run_catboost(label, extra_args):
    model, device = "catboost", "GPU"
    subprocess.run([sys.executable, "-m", "optiml.gpu_check", "--model", model, "--device", device], cwd=REPO, check=True)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + label + "_" + uuid.uuid4().hex[:8]
    local = Path("/content/optiml_gpu_runs") / OWNER / run_id
    local.parent.mkdir(parents=True, exist_ok=True)
    backup = DRIVE_RUNS / run_id
    log_path = local.parent / (run_id + ".log")
    command = [sys.executable, "-u", "-m", "optiml.train", "--prepared-dir", str(PREPARED),
               "--output-dir", str(local), "--model", model, "--device", device,
               "--threads", "2", *extra_args]
    print("Running:", " ".join(command), flush=True)
    process = None
    try:
        with log_path.open("w") as log:
            process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait() != 0:
                raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        if local.exists():
            shutil.copy2(log_path, local / "train.log")
            shutil.copytree(local, backup, dirs_exist_ok=True)
        print("Backup location:", backup)
    return local

## Smoke test

This uses 3% of complete orders to check the full pipeline. Its AUC is not comparable to
the full-data baseline.

In [11]:
smoke = run_catboost("smoke", ["--iterations", "30", "--patience", "10", "--sample-order-fraction", "0.03"])

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261013T183753Z_smoke_00dd2c92 --model catboost --device GPU --threads 2 --iterations 30 --patience 10 --sample-order-fraction 0.03
Loading prepared training features...
Fit 31,352 rows; validate 9,872 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.5583287	best: 0.5583287 (0)	total: 21.5ms	remaining: 623ms
29:	test: 0.6345798	best: 0.6345798 (29)	total: 385ms	remaining: 0us
bestTest = 0.6345798075
bestIteration = 29
{
  "auc": 0.6345799463266852,
  "log_loss": 0.4763081633664319,
  "fit_rows": 31352,
  "validation_rows": 9872,
  "fit_positive_rate": 0.184071191630518,
  "validation_positive_rate": 0.1867909238249595,
  "fit_ids_sha256": "65f92e190ed263a15dbf20a678c956af74451a5a71d1c9d8f087c1dec2dd8462",
  "va

## Full-data CatBoost baseline

Start here after the smoke test. Keep promotion excluded and history enabled, matching the
reported LightGBM baseline. This validates only; it does not produce a Kaggle CSV.

In [12]:
common = ["--cutoff", "2024-07-01", "--iterations", "2000", "--patience", "150"]
cb_baseline = run_catboost("baseline_depth6", common + ["--depth", "6"])
baseline_metrics = json.loads((cb_baseline / "metrics.json").read_text())
print("CatBoost validation AUC:", baseline_metrics["auc"])
print("Validation ID hash:", baseline_metrics["validation_ids_sha256"])

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261013T183811Z_baseline_depth6_c58e08f7 --model catboost --device GPU --threads 2 --cutoff 2024-07-01 --iterations 2000 --patience 150 --depth 6
Loading prepared training features...
Fit 1,048,082 rows; validate 351,923 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6350464	best: 0.6350464 (0)	total: 157ms	remaining: 5m 14s
100:	test: 0.6825500	best: 0.6825500 (100)	total: 3.88s	remaining: 1m 12s
200:	test: 0.6885660	best: 0.6885660 (200)	total: 7.85s	remaining: 1m 10s
300:	test: 0.6921570	best: 0.6921570 (300)	total: 12.7s	remaining: 1m 11s
400:	test: 0.6940526	best: 0.6941180 (377)	total: 16.2s	remaining: 1m 4s
500:	test: 0.6952001	best: 0.6952020 (499)	total: 19.7s	remaining: 59s
600:	test: 0.6955687	best

## Compare focused CatBoost trials

After baseline, set `RUN_TRIALS=True`. Run one GPU fit at a time. Depth 5 tests lower
capacity; depth 8 tests higher capacity; the 12-month window tests recency; dropping raw
entity IDs tests whether memorization is hurting future generalization. Keep the July
validation ID hash identical. Compare October–December AUC, not only the aggregate.

In [13]:
RUN_TRIALS = False
TRIALS = {
    "depth5": ["--depth", "5"],
    "depth8": ["--depth", "8"],
    "last12m": ["--depth", "6", "--train-months", "12"],
    "no_raw_ids": ["--depth", "6", "--drop-features", "customer_id", "product_id", "store_id"],
}
run_paths = {"baseline_depth6": cb_baseline}
trial_specs = {"baseline_depth6": ["--depth", "6"]}
if RUN_TRIALS:
    for name, flags in TRIALS.items():
        run_paths[name] = run_catboost(name, common + flags)
        trial_specs[name] = flags
else:
    print("Review the baseline first, then set RUN_TRIALS=True.")

Review the baseline first, then set RUN_TRIALS=True.


In [14]:
import pandas as pd

results = {name: json.loads((path / "metrics.json").read_text()) for name, path in run_paths.items()}
assert len({m["validation_ids_sha256"] for m in results.values()}) == 1
summary = pd.DataFrame({name: {"auc": m["auc"], "best_iterations": m["best_iterations"],
                               "seconds": m["validation_elapsed_seconds"]}
                        for name, m in results.items()}).T.sort_values("auc", ascending=False)
monthly = pd.DataFrame({name: {month: row["auc"] for month, row in m["monthly_auc"].items()
                               if row["rows"] >= 1000} for name, m in results.items()})
display(summary)
display(monthly)
display(pd.DataFrame(results["baseline_depth6"]["cohorts"]).T)
summary.to_csv(DRIVE_RUNS / (cb_baseline.name + "_comparison.csv"))

,auc,best_iterations,seconds
baseline_depth6,0.703642,1758.0,133.725359


,baseline_depth6
2024-07,0.708816
2024-08,0.709458
2024-09,0.693511
2024-10,0.699136
2024-11,0.707291
2024-12,0.705182


,rows,positives,auc
channel_catalog,24801.0,4660.0,0.634271
channel_marketplace,109469.0,23768.0,0.685320
channel_online,215805.0,34964.0,0.713725
channel_store,1848.0,335.0,0.618565
unseen_customer,67574.0,15607.0,0.687072
seen_customer,284349.0,48120.0,0.702711


## Check recent months for shortlisted models

After comparing July validation, enable this for the best one or two configurations.
It checks drift with an October 2024 cutoff; it is not an independent test set.

In [20]:
RUN_RECENT_CHECK = False
SHORTLIST = summary.index[:2].tolist()
recent_paths = {}
if RUN_RECENT_CHECK:
    for name in dict.fromkeys(["baseline_depth6", *SHORTLIST]):
        recent_paths[name] = run_catboost(name + "_october", ["--cutoff", "2024-10-01", "--iterations", "2000", "--patience", "150"] + trial_specs[name])
    recent_metrics = {name: json.loads((path / "metrics.json").read_text()) for name, path in recent_paths.items()}
    assert len({m["validation_ids_sha256"] for m in recent_metrics.values()}) == 1
    display(pd.DataFrame({name: {"auc": m["auc"], "best_iterations": m["best_iterations"]}
                          for name, m in recent_metrics.items()}).T)

## Recovery diagnostic after a weak leaderboard score

Run this only when temporal validation and Kaggle disagree sharply. It compares the same
three feature policies on two fixed six-month forecast blocks. The stable policy removes
target history and high-cardinality entity identifiers/proxies. Select a policy for another
submission only when it behaves consistently on both blocks; do not optimize another deep
tree configuration against the July block.

In [21]:
RUN_ROLLING_DIAGNOSTIC = True
ROLLING_FOLDS = {
    "jan2024": "2024-01-01",
    "jul2024": "2024-07-01",
}
ROBUST_POLICIES = {
    "entity_history": ["--depth", "5"],
    "no_history": ["--depth", "5", "--no-history"],
    "stable_core": ["--depth", "5", "--no-history", "--drop-features",
                    "customer_id", "product_id", "store_id",
                    "product_supplier_code", "banner_id"],
}
rolling_paths = {}
if RUN_ROLLING_DIAGNOSTIC:
    for fold, cutoff in ROLLING_FOLDS.items():
        for policy, flags in ROBUST_POLICIES.items():
            name = f"{fold}_{policy}"
            rolling_paths[name] = run_catboost(
                name,
                ["--cutoff", cutoff, "--validation-months", "6",
                 "--iterations", "3000", "--patience", "200", *flags],
            )
else:
    print("Set RUN_ROLLING_DIAGNOSTIC=True after recording a weak leaderboard result.")

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261013T184307Z_jan2024_entity_history_5fbb5c09 --model catboost --device GPU --threads 2 --cutoff 2024-01-01 --validation-months 6 --iterations 3000 --patience 200 --depth 5
Loading prepared training features...
Fit 651,761 rows; validate 396,321 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6423150	best: 0.6423150 (0)	total: 99.3ms	remaining: 4m 57s
100:	test: 0.6888583	best: 0.6888583 (100)	total: 4.01s	remaining: 1m 55s
200:	test: 0.6919983	best: 0.6920506 (185)	total: 6.31s	remaining: 1m 27s
300:	test: 0.6928326	best: 0.6932839 (246)	total: 8.62s	remaining: 1m 17s
400:	test: 0.6934900	best: 0.6934900 (400)	total: 10.9s	remaining: 1m 10s
500:	test: 0.6940462	best: 0.6940462 (500)	total: 13.1s	remaining: 

In [22]:
if rolling_paths:
    import pandas as pd

    rolling_metrics = {
        name: json.loads((path / "metrics.json").read_text())
        for name, path in rolling_paths.items()
    }
    rolling_summary = pd.DataFrame({
        name: {
            "auc": metric["auc"],
            "best_iterations": metric["best_iterations"],
            "validation_rows": metric["validation_rows"],
        }
        for name, metric in rolling_metrics.items()
    }).T
    rolling_summary[["fold", "policy"]] = rolling_summary.index.to_series().str.split("_", n=1, expand=True)
    display(rolling_summary.sort_values(["policy", "fold"]))
    display(rolling_summary.pivot(index="policy", columns="fold", values="auc"))

,auc,best_iterations,validation_rows,fold,policy
jan2024_entity_history,0.698323,2049.0,396321.0,jan2024,entity_history
jul2024_entity_history,0.707785,2731.0,351909.0,jul2024,entity_history
jan2024_no_history,0.694315,392.0,396321.0,jan2024,no_history
jul2024_no_history,0.704869,2802.0,351909.0,jul2024,no_history
jan2024_stable_core,0.675602,2868.0,396321.0,jan2024,stable_core
jul2024_stable_core,0.689663,2970.0,351909.0,jul2024,stable_core


fold,jan2024,jul2024
policy,,
entity_history,0.698323,0.707785
no_history,0.694315,0.704869
stable_core,0.675602,0.689663


In [23]:
stable_final = run_catboost(
    "stable_core_final",
    [
        "--cutoff", "2024-07-01",
        "--validation-months", "6",
        "--iterations", "5000",
        "--patience", "300",
        "--depth", "5",
        "--no-history",
        "--drop-features",
        "customer_id",
        "product_id",
        "store_id",
        "product_supplier_code",
        "banner_id",
        "--finalize",
    ],
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "optiml.audit_submission",
        "--csv", str(stable_final / "submission.csv"),
        "--run-dir", str(stable_final),
        "--sample", str(DATA / "sample_submission.csv"),
    ],
    cwd=REPO,
    check=True,
)

stable_metrics = json.loads((stable_final / "metrics.json").read_text())

print("Validation AUC:", stable_metrics["auc"])
print("Best iterations:", stable_metrics["best_iterations"])
print("Submission:", DRIVE_RUNS / stable_final.name / "submission.csv")

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261013T185758Z_stable_core_final_a9c89e5f --model catboost --device GPU --threads 2 --cutoff 2024-07-01 --validation-months 6 --iterations 5000 --patience 300 --depth 5 --no-history --drop-features customer_id product_id store_id product_supplier_code banner_id --finalize
Loading prepared training features...
Fit 1,048,082 rows; validate 351,909 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6067169	best: 0.6067169 (0)	total: 50.3ms	remaining: 4m 11s
100:	test: 0.6513993	best: 0.6513993 (100)	total: 3.54s	remaining: 2m 51s
200:	test: 0.6589505	best: 0.6589505 (200)	total: 8s	remaining: 3m 10s
300:	test: 0.6632290	best: 0.6632290 (300)	total: 11.1s	remaining: 2m 52s
400:	test: 0.6655160	best: 0.6655160 (400)	

In [24]:
stable_metrics = json.loads((stable_final / "metrics.json").read_text())
stable_importance = pd.read_csv(stable_final / "feature_importance.csv")

print("Validation AUC:", stable_metrics["auc"])
print("Best iterations:", stable_metrics["best_iterations"])

display(
    stable_importance
    .sort_values("importance", ascending=False)
    .head(30)
)

display(pd.DataFrame(stable_metrics["monthly_auc"]).T)
display(pd.DataFrame(stable_metrics["cohorts"]).T)

Validation AUC: 0.690251336960769
Best iterations: 4047


,feature,importance
0,customer_store_region_match,6.445220
1,product_packaging_type,5.493858
2,customer_newsletter_opt_in,5.313250
3,gift_wrap,5.243466
4,product_weight_kg,4.567796
5,store_age_days,4.131277
6,store_region,3.930757
7,customer_customer_type,3.757514
8,customer_region,3.576011
9,store_format,3.469737


,rows,positives,auc
2024-07,59589.0,10757.0,0.692712
2024-08,58214.0,10560.0,0.694830
2024-09,55090.0,9937.0,0.680336
2024-10,61345.0,11256.0,0.690071
2024-11,58821.0,10517.0,0.693649
2024-12,58850.0,10698.0,0.691085


,rows,positives,auc
channel_catalog,24801.0,4660.0,0.620393
channel_marketplace,109455.0,23766.0,0.669205
channel_online,215805.0,34964.0,0.701807
channel_store,1848.0,335.0,0.607955
unseen_customer,67564.0,15605.0,0.675128
seen_customer,284345.0,48120.0,0.688795


In [25]:
stable_drops = [
    "customer_id",
    "product_id",
    "store_id",
    "product_supplier_code",
    "banner_id",
]

macro_drops = [
    "cpi",
    "fuel_price",
    "temperature_f",
    "unemployment",
    "fuel_surcharge",
]

age_drops = [
    "customer_tenure_days",
    "customer_tenure_days_invalid",
    "store_age_days",
    "store_age_days_invalid",
]

NEXT_POLICIES = {
    "stable_depth4": [
        "--depth", "4",
        "--no-history",
        "--drop-features", *stable_drops,
    ],
    "stable_no_macro": [
        "--depth", "5",
        "--no-history",
        "--drop-features", *(stable_drops + macro_drops),
    ],
    "stable_no_macro_age": [
        "--depth", "5",
        "--no-history",
        "--drop-features", *(stable_drops + macro_drops + age_drops),
    ],
}

next_paths = {}

for fold, cutoff in {
    "jan2024": "2024-01-01",
    "jul2024": "2024-07-01",
}.items():
    for policy, flags in NEXT_POLICIES.items():
        name = f"{fold}_{policy}"
        next_paths[name] = run_catboost(
            name,
            [
                "--cutoff", cutoff,
                "--validation-months", "6",
                "--iterations", "5000",
                "--patience", "300",
                *flags,
            ],
        )

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261013T191424Z_jan2024_stable_depth4_97ee8773 --model catboost --device GPU --threads 2 --cutoff 2024-01-01 --validation-months 6 --iterations 5000 --patience 300 --depth 4 --no-history --drop-features customer_id product_id store_id product_supplier_code banner_id
Loading prepared training features...
Fit 651,761 rows; validate 396,321 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6043150	best: 0.6043150 (0)	total: 36.4ms	remaining: 3m 2s
100:	test: 0.6550145	best: 0.6550145 (100)	total: 3.44s	remaining: 2m 47s
200:	test: 0.6588566	best: 0.6601352 (197)	total: 6.47s	remaining: 2m 34s
300:	test: 0.6592746	best: 0.6601352 (197)	total: 8.46s	remaining: 2m 12s
400:	test: 0.6615199	best: 0.6615199 (400)	total: 

In [26]:
next_metrics = {
    name: json.loads((path / "metrics.json").read_text())
    for name, path in next_paths.items()
}

next_summary = pd.DataFrame({
    name: {
        "auc": metric["auc"],
        "best_iterations": metric["best_iterations"],
    }
    for name, metric in next_metrics.items()
}).T

next_summary[["fold", "policy"]] = (
    next_summary.index.to_series().str.split("_", n=1, expand=True)
)

display(
    next_summary.pivot(
        index="policy",
        columns="fold",
        values="auc",
    )
)

fold,jan2024,jul2024
policy,,
stable_depth4,0.673597,0.683985
stable_no_macro,0.680338,0.691674
stable_no_macro_age,0.681384,0.691085


In [10]:
from pathlib import Path
import json
import pandas as pd

RUN_ROOT = Path(
    "/content/drive/MyDrive/OptimlRuns/CatBoost/jay"
)

def find_run(label):
    matches = sorted(
        [
            path for path in RUN_ROOT.iterdir()
            if path.is_dir()
            and f"_{label}_" in path.name
            and (path / "COMPLETE.json").exists()
        ],
        key=lambda path: path.name,
    )

    if not matches:
        raise FileNotFoundError(f"No completed Drive run found for {label}")

    selected = matches[-1]
    print(label, "->", selected)
    return selected

next_paths = {
    "jan2024_stable_no_macro": find_run("jan2024_stable_no_macro"),
    "jul2024_stable_no_macro": find_run("jul2024_stable_no_macro"),
    "jan2024_stable_no_macro_age": find_run("jan2024_stable_no_macro_age"),
    "jul2024_stable_no_macro_age": find_run("jul2024_stable_no_macro_age"),
}

jan2024_stable_no_macro -> /content/drive/MyDrive/OptimlRuns/CatBoost/jay/20261013T191918Z_jan2024_stable_no_macro_age_0a8e7c57
jul2024_stable_no_macro -> /content/drive/MyDrive/OptimlRuns/CatBoost/jay/20261013T192753Z_jul2024_stable_no_macro_age_21fea734
jan2024_stable_no_macro_age -> /content/drive/MyDrive/OptimlRuns/CatBoost/jay/20261013T191918Z_jan2024_stable_no_macro_age_0a8e7c57
jul2024_stable_no_macro_age -> /content/drive/MyDrive/OptimlRuns/CatBoost/jay/20261013T192753Z_jul2024_stable_no_macro_age_21fea734


In [12]:
from pathlib import Path
import re

RUN_ROOT = Path(
    "/content/drive/MyDrive/OptimlRuns/CatBoost/jay"
)

def find_run_exact(label):
    pattern = re.compile(
        rf"^\d{{8}}T\d{{6}}Z_{re.escape(label)}_[0-9a-f]{{8}}$"
    )

    matches = sorted(
        path for path in RUN_ROOT.iterdir()
        if path.is_dir()
        and pattern.fullmatch(path.name)
        and (path / "COMPLETE.json").exists()
    )

    if not matches:
        raise FileNotFoundError(f"No completed run found for {label}")

    selected = matches[-1]
    print(label, "->", selected.name)
    return selected

next_paths = {
    "jan2024_stable_no_macro":
        find_run_exact("jan2024_stable_no_macro"),

    "jul2024_stable_no_macro":
        find_run_exact("jul2024_stable_no_macro"),

    "jan2024_stable_no_macro_age":
        find_run_exact("jan2024_stable_no_macro_age"),

    "jul2024_stable_no_macro_age":
        find_run_exact("jul2024_stable_no_macro_age"),
}

jan2024_stable_no_macro -> 20261013T191705Z_jan2024_stable_no_macro_c82dffbb
jul2024_stable_no_macro -> 20261013T192453Z_jul2024_stable_no_macro_391dd6e8
jan2024_stable_no_macro_age -> 20261013T191918Z_jan2024_stable_no_macro_age_0a8e7c57
jul2024_stable_no_macro_age -> 20261013T192753Z_jul2024_stable_no_macro_age_21fea734


In [13]:
from sklearn.metrics import roc_auc_score
import pandas as pd

def compare_stable_blends(fold):
    a = pd.read_parquet(
        next_paths[f"{fold}_stable_no_macro"]
        / "validation_predictions.parquet"
    ).rename(columns={"prediction": "no_macro"})

    b = pd.read_parquet(
        next_paths[f"{fold}_stable_no_macro_age"]
        / "validation_predictions.parquet"
    ).rename(columns={"prediction": "no_macro_age"})

    merged = a.merge(
        b[["order_line_id", "no_macro_age"]],
        on="order_line_id",
        validate="one_to_one",
    )

    rows = []

    for weight_age_removed in [0.0, 0.25, 0.50, 0.75, 1.0]:
        probability_blend = (
            weight_age_removed * merged["no_macro_age"]
            + (1 - weight_age_removed) * merged["no_macro"]
        )

        rank_blend = (
            weight_age_removed
            * merged["no_macro_age"].rank(pct=True)
            + (1 - weight_age_removed)
            * merged["no_macro"].rank(pct=True)
        )

        rows.append({
            "weight_age_removed": weight_age_removed,
            "probability_auc": roc_auc_score(
                merged["returned"], probability_blend
            ),
            "rank_auc": roc_auc_score(
                merged["returned"], rank_blend
            ),
        })

    return pd.DataFrame(rows).set_index("weight_age_removed")

print("January 2024 fold")
display(compare_stable_blends("jan2024"))

print("July 2024 fold")
display(compare_stable_blends("jul2024"))

January 2024 fold


,probability_auc,rank_auc
weight_age_removed,,
0.00,0.680338,0.680338
0.25,0.681338,0.681394
0.50,0.681858,0.681918
0.75,0.681878,0.681912
1.00,0.681384,0.681384


July 2024 fold


,probability_auc,rank_auc
weight_age_removed,,
0.00,0.691674,0.691674
0.25,0.692409,0.692402
0.50,0.692553,0.692542
0.75,0.692105,0.692097
1.00,0.691085,0.691085


In [14]:
from datetime import datetime, timezone
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import roc_auc_score

from optiml.train import write_submission
from optiml.prepare import sha256

stable_drops = [
    "customer_id",
    "product_id",
    "store_id",
    "product_supplier_code",
    "banner_id",
]

macro_drops = [
    "cpi",
    "fuel_price",
    "temperature_f",
    "unemployment",
    "fuel_surcharge",
]

age_drops = [
    "customer_tenure_days",
    "customer_tenure_days_invalid",
    "store_age_days",
    "store_age_days_invalid",
]

common_final = [
    "--cutoff", "2024-07-01",
    "--validation-months", "6",
    "--iterations", "5000",
    "--patience", "300",
    "--depth", "5",
    "--no-history",
]

final_no_macro = run_catboost(
    "stable_no_macro_final",
    common_final + [
        "--drop-features",
        *(stable_drops + macro_drops),
        "--finalize",
    ],
)

final_no_macro_age = run_catboost(
    "stable_no_macro_age_final",
    common_final + [
        "--drop-features",
        *(stable_drops + macro_drops + age_drops),
        "--finalize",
    ],
)

# Check the blend on the final validation predictions.
valid_a = pd.read_parquet(
    final_no_macro / "validation_predictions.parquet"
).rename(columns={"prediction": "prediction_a"})

valid_b = pd.read_parquet(
    final_no_macro_age / "validation_predictions.parquet"
).rename(columns={"prediction": "prediction_b"})

valid = valid_a.merge(
    valid_b[["order_line_id", "prediction_b"]],
    on="order_line_id",
    validate="one_to_one",
)

valid_blend = (
    0.5 * valid["prediction_a"].rank(method="average", pct=True)
    + 0.5 * valid["prediction_b"].rank(method="average", pct=True)
)

print(
    "Final blend validation AUC:",
    roc_auc_score(valid["returned"], valid_blend),
)

# Blend test predictions by rank.
test_a = pd.read_parquet(
    final_no_macro / "test_predictions.parquet"
).rename(columns={"prediction": "prediction_a"})

test_b = pd.read_parquet(
    final_no_macro_age / "test_predictions.parquet"
).rename(columns={"prediction": "prediction_b"})

test_blend = test_a.merge(
    test_b,
    on="order_line_id",
    validate="one_to_one",
)

blended_prediction = (
    0.5 * test_blend["prediction_a"].rank(method="average", pct=True)
    + 0.5 * test_blend["prediction_b"].rank(method="average", pct=True)
)

run_name = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_stable_rank_blend"
)

ensemble_dir = DRIVE_RUNS / run_name
ensemble_dir.mkdir(parents=True, exist_ok=False)

sample = pd.read_csv(DATA / "sample_submission.csv")

write_submission(
    sample,
    test_blend["order_line_id"],
    blended_prediction,
    ensemble_dir / "submission.csv",
)

pd.DataFrame({
    "order_line_id": test_blend["order_line_id"],
    "prediction": blended_prediction,
}).to_parquet(
    ensemble_dir / "test_predictions.parquet",
    index=False,
)

manifest = {
    "model": "50/50 rank blend",
    "component_1": final_no_macro.name,
    "component_2": final_no_macro_age.name,
    "rows": len(test_blend),
    "validation_auc": float(
        roc_auc_score(valid["returned"], valid_blend)
    ),
    "submission_sha256": sha256(
        ensemble_dir / "submission.csv"
    ),
}

(ensemble_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

print(json.dumps(manifest, indent=2))
print("Submission:", ensemble_dir / "submission.csv")

Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261014T020829Z_stable_no_macro_final_4bdb7b48 --model catboost --device GPU --threads 2 --cutoff 2024-07-01 --validation-months 6 --iterations 5000 --patience 300 --depth 5 --no-history --drop-features customer_id product_id store_id product_supplier_code banner_id cpi fuel_price temperature_f unemployment fuel_surcharge --finalize
Loading prepared training features...
Fit 1,048,082 rows; validate 351,909 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6051904	best: 0.6051904 (0)	total: 49.8ms	remaining: 4m 8s
100:	test: 0.6506628	best: 0.6506628 (100)	total: 3.19s	remaining: 2m 34s
200:	test: 0.6584680	best: 0.6584680 (200)	total: 7.83s	remaining: 3m 6s
300:	test: 0.6625892	best: 0.6625892 (300)	total: 10.9s

In [15]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

base_run = find_run_exact("stable_core_final")

stable_drops = [
    "customer_id",
    "product_id",
    "store_id",
    "product_supplier_code",
    "banner_id",
]

seed_runs = {
    "seed42": base_run,
}

for seed in [7, 2026]:
    seed_runs[f"seed{seed}"] = run_catboost(
        f"stable_core_seed{seed}_final",
        [
            "--cutoff", "2024-07-01",
            "--validation-months", "6",
            "--iterations", "5000",
            "--patience", "300",
            "--depth", "5",
            "--seed", str(seed),
            "--no-history",
            "--drop-features", *stable_drops,
            "--finalize",
        ],
    )

validation = None
prediction_columns = []

for name, path in seed_runs.items():
    frame = pd.read_parquet(
        path / "validation_predictions.parquet"
    )[[
        "order_line_id",
        "returned",
        "_month",
        "prediction",
    ]]

    prediction_name = f"prediction_{name}"
    frame = frame.rename(
        columns={"prediction": prediction_name}
    )

    if validation is None:
        validation = frame
    else:
        validation = validation.merge(
            frame[["order_line_id", prediction_name]],
            on="order_line_id",
            validate="one_to_one",
        )

    prediction_columns.append(prediction_name)

for column in prediction_columns:
    validation[f"rank_{column}"] = (
        validation[column].rank(method="average", pct=True)
    )

rank_columns = [
    f"rank_{column}" for column in prediction_columns
]

validation["seed_ensemble"] = (
    validation[rank_columns].mean(axis=1)
)

print("Individual validation AUCs:")
for column in prediction_columns:
    print(
        column,
        roc_auc_score(
            validation["returned"],
            validation[column],
        ),
    )

print(
    "Three-seed rank ensemble AUC:",
    roc_auc_score(
        validation["returned"],
        validation["seed_ensemble"],
    ),
)

monthly = {}
for month in sorted(validation["_month"].unique()):
    mask = validation["_month"].eq(month)
    monthly[month] = roc_auc_score(
        validation.loc[mask, "returned"],
        validation.loc[mask, "seed_ensemble"],
    )

display(pd.Series(monthly, name="ensemble_auc"))

stable_core_final -> 20261013T185758Z_stable_core_final_a9c89e5f
Running: /usr/bin/python3 -u -m optiml.train --prepared-dir /content/optiml_prepared/catboost-baseline-v1 --output-dir /content/optiml_gpu_runs/jay/20261014T023216Z_stable_core_seed7_final_1b67dc0c --model catboost --device GPU --threads 2 --cutoff 2024-07-01 --validation-months 6 --iterations 5000 --patience 300 --depth 5 --seed 7 --no-history --drop-features customer_id product_id store_id product_supplier_code banner_id --finalize
Loading prepared training features...
Fit 1,048,082 rows; validate 351,909 rows. Building model inputs...
Training catboost with backend {'task_type': 'GPU', 'devices': '0'}
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6056317	best: 0.6056317 (0)	total: 51.4ms	remaining: 4m 16s
100:	test: 0.6519556	best: 0.6519556 (100)	total: 3.24s	remaining: 2m 37s
200:	test: 0.6587065	best: 0.6587065 (200)	total: 6.31s	remaining: 2m 30s
300:	test: 0.6631208	best: 0.66312

,ensemble_auc
2024-07,0.690785
2024-08,0.693298
2024-09,0.679369
2024-10,0.688923
2024-11,0.692306
2024-12,0.690108


## Final refit and submission audit

Choose only a validated configuration. Finalization refits on all labeled lines and writes
all 479,995 test predictions. The audit checks order IDs and probabilities before submission.

In [18]:
FINALIZE = False
SELECTED = "baseline_depth6"  # Change after inspecting full and recent validation.
if FINALIZE:
    if SELECTED not in trial_specs:
        raise ValueError("SELECTED has not been evaluated in this session")
    final_run = run_catboost(SELECTED + "_final", common + trial_specs[SELECTED] + ["--finalize"])
    subprocess.run([sys.executable, "-m", "optiml.audit_submission", "--csv", str(final_run / "submission.csv"),
                    "--run-dir", str(final_run), "--sample", str(DATA / "sample_submission.csv")], cwd=REPO, check=True)
    print("Use this exact backed-up CSV:", DRIVE_RUNS / final_run.name / "submission.csv")
else:
    print("Finalization is off during model selection.")

Finalization is off during model selection.
